# VGGish embeddings from raw audio

Reads **mono 16 kHz WAV** files (same format as [`audio_extraction_demucs.ipynb`](audio_extraction_demucs.ipynb): `**/raw/**/<video_stem>_raw.wav`) and writes **128-dim** VGGish vectors per ~0.96 s chunk as `.npy` files.

**Align (1 fps):** duration is taken from each **WAV** (`soundfile`), then chunk embeddings are interpolated to **`ceil(duration) × fps`** frames (default **1 fps**).

**Colab:** mount → install → model → helpers → paths → VGGish cells → align cells.

In [1]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
!pip install -q torch torchaudio numpy resampy soundfile

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.7 MB/s eta 0:00:00


In [3]:
import torch
import numpy as np
import soundfile as sf
from pathlib import Path
from typing import List

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

Device: cuda


In [4]:
vggish = torch.hub.load("harritaylor/torchvggish", "vggish")
vggish = vggish.to(device).eval()
print("VGGish loaded")

/usr/local/lib/python3.12/dist-packages/torch/hub.py:247: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to load(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  _check_repo_is_trusted(


Downloading: "https://github.com/harritaylor/torchvggish/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish-10086976.pth" to /root/.cache/torch/hub/checkpoints/vggish-10086976.pth


100%|██████████| 275M/275M [00:01<00:00, 210MB/s]


Downloading: "https://github.com/harritaylor/torchvggish/releases/download/v0.1/vggish_pca_params-970ea276.pth" to /root/.cache/torch/hub/checkpoints/vggish_pca_params-970ea276.pth


100%|██████████| 177k/177k [00:00<00:00, 6.72MB/s]


VGGish loaded


In [5]:
def extract_vggish_embedding(audio_path: str, model: torch.nn.Module, device: torch.device) -> np.ndarray:
    """Return (num_chunks, 128) embeddings for one file. Model expects 16 kHz mono internally."""
    with torch.inference_mode():
        embeddings = model.forward(audio_path)
    if isinstance(embeddings, torch.Tensor):
        embeddings = embeddings.cpu().numpy()
    return embeddings


def audio_to_vggish_npy(
    audio_path: str,
    output_path: str,
    model: torch.nn.Module,
    device: torch.device,
) -> str:
    """Save embeddings next to mirrored layout; `output_path` is base path (`.npy` added if missing)."""
    embeddings = extract_vggish_embedding(audio_path, model, device)
    out = Path(output_path)
    if out.suffix != ".npy":
        out = out.with_suffix(".npy")
    out.parent.mkdir(parents=True, exist_ok=True)
    np.save(str(out), embeddings)
    print(f"Saved {out}  shape={embeddings.shape}  (~{embeddings.shape[0]} × 0.96s)")
    return str(out)


def batch_audio_to_embeddings(
    audio_dir: str,
    output_dir: str,
    model: torch.nn.Module,
    device: torch.device,
    audio_extensions: tuple = (".wav",),
    preserve_structure: bool = True,
) -> List[str]:
    """
    Walk `audio_dir` recursively (e.g. `.../audio_output/<league>/raw/` from extraction notebook).
    Writes `.npy` files under `output_dir` with the same relative folders; stem matches the WAV (e.g. `2_224p_raw.npy`).
    """
    audio_dir = Path(audio_dir)
    paths = []
    for ext in audio_extensions:
        paths.extend(audio_dir.rglob(f"*{ext}"))
    print(f"Found {len(paths)} WAV files under {audio_dir}")
    saved = []
    for i, audio_path in enumerate(paths):
        rel = audio_path.relative_to(audio_dir)
        print(f"\n[{i + 1}/{len(paths)}] {rel}")
        try:
            if preserve_structure:
                out_base = Path(output_dir) / rel.parent / rel.stem
            else:
                out_base = Path(output_dir) / rel.stem
            saved.append(audio_to_vggish_npy(str(audio_path), str(out_base), model, device))
        except Exception as e:
            print(f"Error: {audio_path}: {e}")
    print(f"\nDone: {len(saved)} npy files")
    return saved

## Paths

Set **`AUDIO_OUTPUT_BASE`** to the same **`OUTPUT_DIR`** you used in `audio_extraction_demucs.ipynb`. Raw WAVs live under **`{AUDIO_OUTPUT_BASE}/raw/`**.

In [6]:
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/data_sn/audio_output")

LEAGUES = [
    "england_epl",
    "france_ligue-1",
    "germany_bundesliga",
    "italy_serie-a",
    "spain_laliga",
    "europe_uefa-champions-league",
]

SEASONS = ["2014-2015", "2015-2016", "2016-2017", "2019-2020"]

VIDEO_FPS = 1.0

# Preview what will be processed
for league in LEAGUES:
    for season in SEASONS:
        season_dir = DATA_ROOT / league / season
        wavs = sorted(season_dir.rglob("*_raw.wav")) if season_dir.exists() else []
        print(f"{league}/{season}: {len(wavs)} *_raw.wav files")

england_epl/2014-2015: 12 *_raw.wav files
england_epl/2015-2016: 98 *_raw.wav files
england_epl/2016-2017: 98 *_raw.wav files
england_epl/2019-2020: 0 *_raw.wav files
france_ligue-1/2014-2015: 2 *_raw.wav files
france_ligue-1/2015-2016: 6 *_raw.wav files
france_ligue-1/2016-2017: 86 *_raw.wav files
france_ligue-1/2019-2020: 0 *_raw.wav files
germany_bundesliga/2014-2015: 16 *_raw.wav files
germany_bundesliga/2015-2016: 36 *_raw.wav files
germany_bundesliga/2016-2017: 70 *_raw.wav files
germany_bundesliga/2019-2020: 0 *_raw.wav files
italy_serie-a/2014-2015: 22 *_raw.wav files
italy_serie-a/2015-2016: 18 *_raw.wav files
italy_serie-a/2016-2017: 170 *_raw.wav files
italy_serie-a/2019-2020: 0 *_raw.wav files
spain_laliga/2014-2015: 36 *_raw.wav files
spain_laliga/2015-2016: 72 *_raw.wav files
spain_laliga/2016-2017: 126 *_raw.wav files
spain_laliga/2019-2020: 16 *_raw.wav files
europe_uefa-champions-league/2014-2015: 74 *_raw.wav files
europe_uefa-champions-league/2015-2016: 90 *_raw.wav 

### Single file (VGGish chunks `.npy`)

In [7]:
rel_wav = SAMPLE_RAW_WAV.relative_to(RAW_AUDIO_DIR)
out_base = (EMBEDDING_OUTPUT_DIR / rel_wav).with_suffix("")

out_npy = audio_to_vggish_npy(str(SAMPLE_RAW_WAV), str(out_base), vggish, device)

NameError: name 'SAMPLE_RAW_WAV' is not defined

### Batch (all `*_raw.wav` under `raw/` → chunk `.npy`)

In [8]:
all_npy_paths = []

for league in LEAGUES:
    for season in SEASONS:
        season_dir = DATA_ROOT / league / season
        if not season_dir.exists():
            print(f"[SKIP] {league}/{season} — folder not found")
            continue

        embedding_dir = season_dir / "embeddings" / "vggish"
        raw_wavs = sorted(season_dir.rglob("*_raw.wav"))

        # Exclude any wavs already inside the embeddings folder
        raw_wavs = [w for w in raw_wavs if "embeddings" not in w.parts]

        print(f"\n=== {league}/{season}: {len(raw_wavs)} files ===")
        for i, wav in enumerate(raw_wavs):
            rel = wav.relative_to(season_dir)
            print(f"  [{i + 1}/{len(raw_wavs)}] {rel}")
            try:
                out_base = embedding_dir / rel.parent / rel.stem
                all_npy_paths.append(audio_to_vggish_npy(str(wav), str(out_base), vggish, device))
            except Exception as e:
                print(f"    Error: {e}")

print(f"\n=== TOTAL: {len(all_npy_paths)} npy files saved ===")


=== england_epl/2014-2015: 12 files ===
  [1/12] raw/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/1_224p_raw.wav
Saved /content/drive/MyDrive/data_sn/audio_output/england_epl/2014-2015/embeddings/vggish/raw/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/1_224p_raw.npy  shape=(2812, 128)  (~2812 × 0.96s)
  [2/12] raw/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/2_224p_raw.wav
Saved /content/drive/MyDrive/data_sn/audio_output/england_epl/2014-2015/embeddings/vggish/raw/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/2_224p_raw.npy  shape=(2812, 128)  (~2812 × 0.96s)
  [3/12] raw/2015-02-21 - 18-00 Crystal Palace 1 - 2 Arsenal/1_224p_raw.wav
Saved /content/drive/MyDrive/data_sn/audio_output/england_epl/2014-2015/embeddings/vggish/raw/2015-02-21 - 18-00 Crystal Palace 1 - 2 Arsenal/1_224p_raw.npy  shape=(2882, 128)  (~2882 × 0.96s)
  [4/12] raw/2015-02-21 - 18-00 Crystal Palace 1 - 2 Arsenal/2_224p_raw.wav
Saved /content/drive/MyDrive/data_sn/audio_output/england_epl/2014-2015/embeddings/vggish/raw/2015-02-21

### Align to `VIDEO_FPS` using WAV duration

Reads each **raw WAV** with `soundfile` for **exact duration**, loads the matching **chunk** `.npy` from `EMBEDDING_OUTPUT_DIR`, interpolates to **`int(duration * VIDEO_FPS)`** frames (at least **1**), saves under **`ALIGNED_1FPS_DIR`** with the same relative path.

In [9]:
def wav_duration_seconds(wav_path: str) -> float:
    """Length of the WAV in seconds (from file header / samples)."""
    return float(sf.info(wav_path).duration)


def align_audio_to_video_features(
    audio_embeddings: np.ndarray,
    video_duration_seconds: float,
    video_fps: float = 1.0,
    vggish_chunk_duration: float = 0.96,
) -> np.ndarray:
    num_chunks = audio_embeddings.shape[0]
    num_frames = max(1, int(video_duration_seconds * video_fps))
    audio_times = np.arange(num_chunks) * vggish_chunk_duration + vggish_chunk_duration / 2
    video_times = np.arange(num_frames) / video_fps
    aligned = np.zeros((num_frames, 128))
    for dim in range(128):
        aligned[:, dim] = np.interp(video_times, audio_times, audio_embeddings[:, dim])
    return aligned


def align_vggish_from_wav(
    wav_path: str,
    chunk_embeddings: np.ndarray,
    video_fps: float = 1.0,
    vggish_chunk_duration: float = 0.96,
) -> np.ndarray:
    """Use WAV length as timeline length (same clip as extracted audio)."""
    dur = wav_duration_seconds(wav_path)
    return align_audio_to_video_features(
        chunk_embeddings, dur, video_fps, vggish_chunk_duration
    )


def batch_align_vggish_to_fps(
    raw_audio_dir: str,
    chunk_npy_dir: str,
    aligned_npy_dir: str,
    video_fps: float = 1.0,
) -> List[str]:
    raw_audio_dir = Path(raw_audio_dir)
    chunk_npy_dir = Path(chunk_npy_dir)
    aligned_npy_dir = Path(aligned_npy_dir)
    wavs = sorted(raw_audio_dir.rglob("*.wav"))
    print(f"Aligning {len(wavs)} files at {video_fps} fps")
    saved = []
    for i, wav in enumerate(wavs):
        rel = wav.relative_to(raw_audio_dir)
        chunk_npy = chunk_npy_dir / rel.parent / f"{wav.stem}.npy"
        if not chunk_npy.is_file():
            print(f"Skip (no chunks): {rel}")
            continue
        print(f"[{i + 1}/{len(wavs)}] {rel}  dur={wav_duration_seconds(str(wav)):.2f}s")
        try:
            emb = np.load(chunk_npy)
            aligned = align_vggish_from_wav(str(wav), emb, video_fps=video_fps)
            out = aligned_npy_dir / rel.parent / f"{wav.stem}.npy"
            out.parent.mkdir(parents=True, exist_ok=True)
            np.save(str(out), aligned)
            saved.append(str(out))
            print(f"  -> {aligned.shape}")
        except Exception as e:
            print(f"Error {wav}: {e}")
    print(f"Done: {len(saved)} aligned npy")
    return saved

In [ ]:
rel_wav = SAMPLE_RAW_WAV.relative_to(RAW_AUDIO_DIR)
chunk_npy = (EMBEDDING_OUTPUT_DIR / rel_wav).with_suffix(".npy")
chunks = np.load(chunk_npy)
aligned = align_vggish_from_wav(str(SAMPLE_RAW_WAV), chunks, video_fps=VIDEO_FPS)
aligned_path = (ALIGNED_1FPS_DIR / rel_wav).with_suffix(".npy")
aligned_path.parent.mkdir(parents=True, exist_ok=True)
np.save(str(aligned_path), aligned)
print(f"Saved {aligned_path}  shape={aligned.shape}")

In [10]:
all_aligned_paths = []

for league in LEAGUES:
    for season in SEASONS:
        season_dir = DATA_ROOT / league / season
        if not season_dir.exists():
            print(f"[SKIP] {league}/{season} — folder not found")
            continue

        embedding_dir = season_dir / "embeddings" / "vggish"
        aligned_dir   = season_dir / "embeddings" / "vggish_1fps"
        raw_wavs = sorted(season_dir.rglob("*_raw.wav"))
        raw_wavs = [w for w in raw_wavs if "embeddings" not in w.parts]

        print(f"\n=== {league}/{season}: aligning {len(raw_wavs)} files at {VIDEO_FPS} fps ===")
        for i, wav in enumerate(raw_wavs):
            rel = wav.relative_to(season_dir)
            chunk_npy = embedding_dir / rel.parent / f"{wav.stem}.npy"
            if not chunk_npy.is_file():
                print(f"  Skip (no chunks yet): {rel}")
                continue
            print(f"  [{i + 1}/{len(raw_wavs)}] {rel}  dur={wav_duration_seconds(str(wav)):.2f}s")
            try:
                emb = np.load(chunk_npy)
                aligned = align_vggish_from_wav(str(wav), emb, video_fps=VIDEO_FPS)
                out = aligned_dir / rel.parent / f"{wav.stem}.npy"
                out.parent.mkdir(parents=True, exist_ok=True)
                np.save(str(out), aligned)
                all_aligned_paths.append(str(out))
                print(f"    -> {aligned.shape}")
            except Exception as e:
                print(f"    Error: {e}")

print(f"\n=== TOTAL: {len(all_aligned_paths)} aligned npy files saved ===")


=== england_epl/2014-2015: aligning 12 files at 1.0 fps ===
  [1/12] raw/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/1_224p_raw.wav  dur=2700.01s
    -> (2700, 128)
  [2/12] raw/2015-02-21 - 18-00 Chelsea 1 - 1 Burnley/2_224p_raw.wav  dur=2700.02s
    -> (2700, 128)
  [3/12] raw/2015-02-21 - 18-00 Crystal Palace 1 - 2 Arsenal/1_224p_raw.wav  dur=2767.02s
    -> (2767, 128)
  [4/12] raw/2015-02-21 - 18-00 Crystal Palace 1 - 2 Arsenal/2_224p_raw.wav  dur=2974.02s
    -> (2974, 128)
  [5/12] raw/2015-02-21 - 18-00 Swansea 2 - 1 Manchester United/1_224p_raw.wav  dur=2725.00s
    -> (2725, 128)
  [6/12] raw/2015-02-21 - 18-00 Swansea 2 - 1 Manchester United/2_224p_raw.wav  dur=2937.00s
    -> (2937, 128)
  [7/12] raw/2015-02-22 - 19-15 Southampton 0 - 2 Liverpool/1_224p_raw.wav  dur=2700.00s
    -> (2700, 128)
  [8/12] raw/2015-02-22 - 19-15 Southampton 0 - 2 Liverpool/2_224p_raw.wav  dur=2801.00s
    -> (2801, 128)
  [9/12] raw/2015-04-11 - 19-30 Burnley 0 - 1 Arsenal/1_224p_raw.wav  dur=270

## Recap

- **Chunks:** `(num_chunks, 128)` at ~0.96 s — saved under `embeddings/vggish/`.
- **Aligned:** shape `(max(1, int(T * fps)), 128)` where **T** is **WAV duration** in seconds — saved under `embeddings/vggish_1fps/` (folder name is fixed; set `VIDEO_FPS` in the paths cell).